# Feature Engineering and Target Construction

This notebook isolates target creation and modeling-table construction. The current cells preserve the existing baseline behavior and add the first ticker-hour aggregation needed for the next modeling pass.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import load_merged_data, summarize_data_quality
from src.plots import set_plot_style

set_plot_style()

from src.feature_engineering import add_next_close_target, build_clean_hourly_dataset
from src.config import CLEAN_FINANCE_START, MODELING_DATASET_PATH, PROCESSED_DATA_DIR

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

df = load_merged_data()
display(df.head())


Ticker,Timestamp,Text,Sentiment,Post_Count,Open,High,Low,Close,Volume
str,"datetime[μs, UTC]",str,f64,i64,f64,f64,f64,f64,i64
"""$AAPL""",2025-07-07 23:00:00 UTC,"""BREAKING: Apple’s, $AAPL, top …",-0.220868,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-07 23:00:00 UTC,"""🔴 Importance: 65/100 #news #To…",-0.440318,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-08 00:00:00 UTC,"""What do you think #investors? …",0.150104,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-08 00:00:00 UTC,"""Lemme guess... you bought AAPL…",0.012221,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-08 02:00:00 UTC,"""$AAPL Support / Resistance Lev…",0.027583,1,210.130005,210.490005,208.449997,209.279999,9381795


## Current Baseline Target

This is the target used in the original notebook: `1` when the next close for the same ticker is greater than the current close. It is kept here so the baseline remains reproducible.

In [2]:
baseline_df = add_next_close_target(df)

print("Post-level target balance:")
print(baseline_df["Target_Direction"].value_counts().sort("Target_Direction"))


Post-level target balance:
shape: (2, 2)
┌──────────────────┬────────┐
│ Target_Direction ┆ count  │
│ ---              ┆ ---    │
│ i32              ┆ u32    │
╞══════════════════╪════════╡
│ 0                ┆ 187033 │
│ 1                ┆ 7835   │
└──────────────────┴────────┘


## Step 1: Clean Ticker-Hour Modeling Input

The raw dataset has one row per post. For final modeling, one row per ticker-hour is cleaner because every post in the same ticker-hour shares the same market price target.

This step also filters out old social posts before the reliable Yahoo Finance hourly coverage window. The current cutoff is `2024-06-01`.

In [3]:
clean_hourly_df = build_clean_hourly_dataset(df)

print(f"Finance coverage cutoff: {CLEAN_FINANCE_START:%Y-%m-%d}")
print(f"Raw rows: {df.height:,}")
print(f"Clean ticker-hour rows: {clean_hourly_df.height:,}")
print(f"Unique tickers: {clean_hourly_df.select('Ticker').n_unique()}")
display(clean_hourly_df.head())


Finance coverage cutoff: 2024-06-01
Raw rows: 194,891
Clean ticker-hour rows: 54,572
Unique tickers: 23


Ticker,Timestamp,Open,High,Low,Close,Volume,Sentiment_Mean,Sentiment_Median,Sentiment_Std,Sentiment_Min,Sentiment_Max,Positive_Post_Count,Negative_Post_Count,Neutral_Post_Count,Post_Count,Bullishness_Index
str,"datetime[μs, UTC]",f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,u32,u32,u32,u32,f64
"""$AAPL""",2025-07-07 23:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,-0.330593,-0.330593,0.155175,-0.440318,-0.220868,0,2,0,2,2.1475e9
"""$AAPL""",2025-07-08 00:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,0.081162,0.081162,0.097498,0.012221,0.150104,1,0,1,2,0.5
"""$AAPL""",2025-07-08 02:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,0.027583,0.027583,0.0,0.027583,0.027583,0,0,1,1,0.0
"""$AAPL""",2025-07-08 05:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,-0.033409,0.075754,0.479426,-0.558003,0.382022,2,1,0,3,0.333333
"""$AAPL""",2025-07-08 06:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,0.019368,0.019368,0.0,0.019368,0.019368,0,0,1,1,0.0


## Step 1 Validation

These checks confirm that the clean modeling input has one row per ticker-hour, no nulls in critical fields, and timestamps inside the valid finance window.

In [4]:
duplicate_ticker_hours = clean_hourly_df.height - clean_hourly_df.select(["Ticker", "Timestamp"]).unique().height

print(f"Duplicate ticker-hour rows: {duplicate_ticker_hours}")
print("Timestamp range:")
display(clean_hourly_df.select(pl.col("Timestamp").min().alias("min_ts"), pl.col("Timestamp").max().alias("max_ts")))

print("Null counts:")
print(clean_hourly_df.null_count())

print("Ticker-hour rows by ticker:")
display(clean_hourly_df.group_by("Ticker").len().sort("len", descending=True))


Duplicate ticker-hour rows: 0
Timestamp range:


min_ts,max_ts
"datetime[μs, UTC]","datetime[μs, UTC]"
2024-06-01 15:00:00 UTC,2026-04-28 23:00:00 UTC


Null counts:
shape: (1, 17)
┌────────┬───────────┬──────┬──────┬───┬───────────────┬───────────────┬────────────┬──────────────┐
│ Ticker ┆ Timestamp ┆ Open ┆ High ┆ … ┆ Negative_Post ┆ Neutral_Post_ ┆ Post_Count ┆ Bullishness_ │
│ ---    ┆ ---       ┆ ---  ┆ ---  ┆   ┆ _Count        ┆ Count         ┆ ---        ┆ Index        │
│ u32    ┆ u32       ┆ u32  ┆ u32  ┆   ┆ ---           ┆ ---           ┆ u32        ┆ ---          │
│        ┆           ┆      ┆      ┆   ┆ u32           ┆ u32           ┆            ┆ u32          │
╞════════╪═══════════╪══════╪══════╪═══╪═══════════════╪═══════════════╪════════════╪══════════════╡
│ 0      ┆ 0         ┆ 0    ┆ 0    ┆ … ┆ 0             ┆ 0             ┆ 0          ┆ 0            │
└────────┴───────────┴──────┴──────┴───┴───────────────┴───────────────┴────────────┴──────────────┘
Ticker-hour rows by ticker:


Ticker,len
str,u32
"""$PLTR""",5055
"""$MSTR""",4823
"""$GOOGL""",4530
"""$GME""",4522
"""$MSFT""",4190
…,…
"""$HOOD""",378
"""$COIN""",337
"""$SPY""",324


## Save Step 1 Modeling Input

This table is the clean targetless modeling input for the next step. Step 2 will add the final target definition and may overwrite this artifact with a target-bearing version.

In [5]:
clean_hourly_df.write_csv(MODELING_DATASET_PATH)
print(f"Saved {clean_hourly_df.height:,} rows to {MODELING_DATASET_PATH}")


Saved 54,572 rows to /home/laytc/GitHubRepo/Penn/cis2450/cis-2450-final-proj/data/processed/modeling_dataset.csv


## Next Feature Engineering Work

- Add intraday/overnight target logic.
- Add lag returns, rolling returns, volume anomaly, sentiment EMA, sentiment z-score, hour/day features, and ticker encoding.
- Apply any scaling or sampling only after the train/test split in the modeling notebook.